# S06-13A (Student) — Building Graph from Rhino OBJ

Model the building in **Rhino**. Export **four OBJ files**:

| File | Layer |
|------|--------|
| `ground.obj` | slab / podium |
| `columns.obj` | columns |
| `offices.obj` | office volumes |
| `core.obj` | core + corridors |

Store them in `Supporting Files/` (or change `SUPPORT_DIR` in the setup cell).

---

## Pipeline

1. **Import** OBJs → `Topology.ByOBJPath`
2. **Tag** each cell (`cell_type` selectors)
3. **CellComplex** → merge all cells (`Topology.SelfMerge`)
4. **Transfer** dictionaries onto cells
5. **Graph** → `Graph.ByTopology` + one-hot features
6. **Export** CSV → `Graph.ExportToCSV`
7. **Predict** → **S06-13** (Phase 2), `pyg_model.pt`


Reference outputs (gray) below = example building; yours will differ.


### Setup

Install `topologicpy` if needed. Import `Topology`, `Cluster`, `Dictionary`, `Graph`, `Helper`.

In [1]:
# !pip install topologicpy

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Color import Color
from topologicpy.Helper import Helper


e:\IAAC Local GIT Repositories\Graph ML - Environment\.env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Import parts

`Topology.ByOBJPath` for each OBJ. **TODO:** your four paths.

In [2]:
# TODO: Import geometry, use code below as an example

ground_objs = Topology.ByOBJPath(r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Assignment-03_RamonGarcia\geometries\obj_geometry_v2.0\ground.obj", transposeAxes=False)
column_objs = Topology.ByOBJPath(r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Assignment-03_RamonGarcia\geometries\obj_geometry_v2.0\columns.obj", transposeAxes=False)
office_objs = Topology.ByOBJPath(r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Assignment-03_RamonGarcia\geometries\obj_geometry_v2.0\offices.obj", transposeAxes=False)
core_objs = Topology.ByOBJPath(r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Assignment-03_RamonGarcia\geometries\obj_geometry_v2.0\core.obj", transposeAxes=False)


**Check:** `Topology.Show(ground_objs, office_objs, core_objs, column_objs)`

In [3]:
# Topology.Show(ground_objs, office_objs, core_objs, column_objs)

### 2. One node per closed geometry

Per layer: Faces → Flatten → SelfMerge → `Topology.Cells` (each closed volume = **one cell**).
For each cell store a `Dictionary` (`cell_type`, `cell_name`, `cell_color`) on its **center** (`InternalVertex`).
These centers are the graph nodes — corners / mesh vertices are never used.

In [ ]:
# One node per CLOSED geometry: tag each cell and take its center (InternalVertex).
# Mapping: ground = 0, office = 1, column = 2, core = 3

def layer_cells(objs):
    faces = [Topology.Faces(o) for o in objs if Topology.IsInstance(o, "Topology")]
    faces = Helper.Flatten(faces)
    merged = Topology.SelfMerge(Cluster.ByTopologies(faces))
    return Topology.Cells(merged)

ground_cells = layer_cells(ground_objs)
office_cells = layer_cells(office_objs)
column_cells = layer_cells(column_objs)
core_cells   = layer_cells(core_objs)

layers = [
    (ground_cells, 0, "ground", "red"),
    (office_cells, 1, "office", "green"),
    (column_cells, 2, "column", "blue"),
    (core_cells,   3, "core",   "purple"),
]

cells = []           # one closed geometry per entry
node_vertices = []   # its center, carrying the dictionary
for layer, cell_type, cell_name, cell_color in layers:
    for cell in layer:
        d = Dictionary.ByKeysValues(["cell_type", "cell_name", "cell_color"],
                                    [cell_type, cell_name, cell_color])
        v = Topology.InternalVertex(cell)
        v = Topology.SetDictionary(v, d)
        cells.append(cell)
        node_vertices.append(v)

print("closed geometries (nodes):", len(cells))
print("  ground:", len(ground_cells),
      "| office:", len(office_cells),
      "| column:", len(column_cells),
      "| core:", len(core_cells))

### 3. Adjacency edges

Connect two cells when they **touch** (boundary distance ≈ 0).

> We do **not** `SelfMerge` all cells into a single `CellComplex`: that would slice every shared
> face and explode 334 closed geometries into ~1600 fragment-cells (the source of the extra nodes).
> Instead we keep one node per geometry and add an edge for each touching pair. A cheap bounding-box
> test pre-filters candidates so `Topology.ShortestDistance` only runs where boxes overlap.

In [ ]:
from itertools import combinations

def bbox(cell):
    vs = Topology.Vertices(cell)
    xs = [Vertex.X(v) for v in vs]
    ys = [Vertex.Y(v) for v in vs]
    zs = [Vertex.Z(v) for v in vs]
    return (min(xs), min(ys), min(zs), max(xs), max(ys), max(zs))

boxes     = [bbox(c) for c in cells]
pad       = 0.05    # tolerance for "bounding boxes overlap"
touch_tol = 0.01    # tolerance for "cells touch"

def boxes_overlap(a, b):
    return (a[0] <= b[3] + pad and b[0] <= a[3] + pad and
            a[1] <= b[4] + pad and b[1] <= a[4] + pad and
            a[2] <= b[5] + pad and b[2] <= a[5] + pad)

graph_edges = []
for i, j in combinations(range(len(cells)), 2):
    if not boxes_overlap(boxes[i], boxes[j]):
        continue
    d = Topology.ShortestDistance(cells[i], cells[j])
    if d is not None and d < touch_tol:
        e = Edge.ByStartVertexEndVertex(node_vertices[i], node_vertices[j], silent=True)
        if e is not None:
            graph_edges.append(e)

print("edges (touching cells):", len(graph_edges))

### 4. Build graph

`Graph.ByVerticesEdges` — nodes are the cell centers (carrying their dictionaries), edges are the touching pairs.

In [ ]:
graph = Graph.ByVerticesEdges(node_vertices, graph_edges, silent=True)
print(graph)
print("graph nodes:", len(Graph.Vertices(graph)), "| graph edges:", len(Graph.Edges(graph)))

### 5. Node features

One-hot `feature_00`…`feature_03` from each node's `cell_type`.

In [ ]:
def one_hot_encode(value, n):
    if value < 0 or value >= n:
        raise ValueError(f"Value {value} is out of range [0, {n-1}]")

    result = [0] * n
    result[value] = 1
    return result

vertices = Graph.Vertices(graph)

# ground = 0
# office = 1
# column = 2
# core = 3

feature_names = ["feature_" + str(i).zfill(2) for i in range(4)]
print(feature_names)

for v in vertices:
    d = Topology.Dictionary(v)
    cell_type = Dictionary.ValueAtKey(d, "cell_type")
    if cell_type is None:
        continue            # node without a category: skip
    ohe = one_hot_encode(cell_type, 4)
    for i, feature_name in enumerate(feature_names):
        d = Dictionary.SetValueAtKey(d, feature_name, ohe[i])
    v = Topology.SetDictionary(v, d)

Topology.Show(graph,
              vertexColorKey="cell_color",
              backgroundColor="white",
              edgeWidthKey="width",
              edgeColorKey="color")

### 6. Export CSV

`Graph.ExportToCSV` → folder with `graphs.csv`, `nodes.csv`, `edges.csv`.

In [ ]:
status = Graph.ExportToCSV(graph,
                           path=r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Assignment-03_RamonGarcia\dataset_graph_classification",
                           nodeFeaturesKeys=feature_names,
                           overwrite=True)
print(status)

True


### 7. Predict (S06-13)

**S06-13 GML Graph Classification** → Phase 2: set `dataset_dir` to your export folder, `LoadModel(pyg_model.pt)`, `Predict()` → label 0–4.